# J.A.R.V.I.S wake-phrase trainer (pre-patched)

Trains a custom [openWakeWord](https://github.com/dscripka/openWakeWord) wake-phrase model.
This is the official `automatic_model_training.ipynb` with fixes baked in for current
Colab images (Python 3.12, torch 2.6+, numpy 2, pyarrow 17+):

- `piper-sample-generator` pinned to **v2.0.0** (upstream restructured; `train.py` needs the old layout)
- `piper-phonemize` installed via community wheels (no official Python 3.12 build)
- `torch-audiomentations` upgraded + `torchaudio.set_audio_backend` shim
- `torch.load(weights_only=False)` patch for torch 2.6's stricter default
- `numpy 1.26 / pyarrow 16.1 / datasets 2.14` compatible set
- TensorFlow / onnx_tf pins **dropped** (they no longer install; only needed for
  `.tflite` export — J.A.R.V.I.S runs the `.onnx`)
- Training steps report errors loudly instead of failing silently
- The trained `.onnx` auto-downloads at the end

**Usage:** `Runtime → Change runtime type → T4 GPU` → edit `PHRASE` in the config cell
→ `Runtime → Run all` (~45–75 min) → the browser downloads the model when done.


# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup — patched for current Colab images
## MUST run before anything imports numpy/torch (it downgrades numpy on disk;
## a kernel that already loaded numpy 2.x would binary-clash afterwards).
import sys
if "numpy" in sys.modules:
    print("!" * 70)
    print("numpy is already loaded in this kernel. After this cell finishes:")
    print("Runtime > Restart session, then Run all (top to bottom).")
    print("!" * 70)

# piper-sample-generator at v2.0.0: the layout openwakeword's train.py expects
!git clone --depth 1 --branch v2.0.0 https://github.com/rhasspy/piper-sample-generator
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# piper-phonemize has no official Python 3.12 wheels; use community build, fall back to official
!pip install -q piper-phonemize-cross || pip install -q piper-phonemize
!pip install -q webrtcvad

# openwakeword (full training install)
!git clone https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword

# Pinned deps. NOTE: the original notebook's tensorflow-cpu==2.8.1 /
# tensorflow_probability / onnx_tf pins are dropped on purpose — they don't
# install on Python 3.12 and are only needed for .tflite export.
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 acoustics==0.2.6 pronouncing==0.2.0 deep-phonemizer==0.0.19 datasets==2.14.6
!pip install -q "numpy==1.26.4" "pyarrow==16.1.0"
!pip install -qU torch-audiomentations

# Shims for APIs removed by newer torch/torchaudio
!sed -i 's/torchaudio\.set_audio_backend(.*)/pass/' /usr/local/lib/python3*/dist-packages/torch_audiomentations/utils/io.py
!sed -i 's/torch\.load(\([^)]*\))/torch.load(\1, weights_only=False)/g' piper-sample-generator/generate_samples.py

# openwakeword's shared feature models (Colab workaround)
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# Sanity checks — fail HERE, loudly, not 40 minutes into training
import importlib
for mod in ("piper_phonemize", "webrtcvad", "torch_audiomentations"):
    importlib.import_module(mod); print("ok:", mod)
assert os.path.exists("piper-sample-generator/generate_samples.py"), "piper-sample-generator has the wrong layout"
print("ENVIRONMENT READY")


In [ ]:
# Fail fast if the runtime isn't a GPU (Runtime > Change runtime type > T4 GPU)
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then Run all again."
!nvidia-smi -L


In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Convert audioset files to 16khz sample rate
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break


In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

In [ ]:
# ============== EDIT THIS: one phrase per training run ==============
PHRASE = "wake up mommys home"        # second run: "jarvis you there"
# No apostrophes/punctuation — they break file paths downstream.
# =====================================================================

config["target_phrase"] = [PHRASE]
config["model_name"] = PHRASE.replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
config["piper_sample_generator_path"] = "./piper-sample-generator"

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)
print("Training config written for:", config["model_name"])


# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: generate synthetic clips (~10-25 min, silent while running)
import subprocess, sys, time
_t0 = time.time()
r = subprocess.run([sys.executable, "openwakeword/openwakeword/train.py",
                    "--training_config", "my_model.yaml", "--generate_clips"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
print(f"[--generate_clips took {time.time()-_t0:.0f}s]")
if r.returncode != 0:
    print("---- STDERR (tail) ----")
    print(r.stderr[-4000:])
    raise RuntimeError("--generate_clips FAILED — see stderr above")
print("--generate_clips OK")


In [ ]:
# Step 2: augment clips (~5-10 min)
import subprocess, sys, time
_t0 = time.time()
r = subprocess.run([sys.executable, "openwakeword/openwakeword/train.py",
                    "--training_config", "my_model.yaml", "--augment_clips"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
print(f"[--augment_clips took {time.time()-_t0:.0f}s]")
if r.returncode != 0:
    print("---- STDERR (tail) ----")
    print(r.stderr[-4000:])
    raise RuntimeError("--augment_clips FAILED — see stderr above")
print("--augment_clips OK")


In [ ]:
# Step 3: train the model (~10-30 min)
import subprocess, sys, time
_t0 = time.time()
r = subprocess.run([sys.executable, "openwakeword/openwakeword/train.py",
                    "--training_config", "my_model.yaml", "--train_model"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
print(f"[--train_model took {time.time()-_t0:.0f}s]")
if r.returncode != 0:
    print("---- STDERR (tail) ----")
    print(r.stderr[-4000:])
    raise RuntimeError("--train_model FAILED — see stderr above")
print("--train_model OK")


In [ ]:
# Download the trained model — J.A.R.V.I.S uses the .onnx (no tflite needed)
import os
from google.colab import files
path = f"my_custom_model/{config['model_name']}.onnx"
assert os.path.exists(path), f"{path} is missing — check the training cells above"
print(path, "-", os.path.getsize(path)//1024, "KB")
files.download(path)


## Next steps

1. Put the downloaded `.onnx` into `models/wake/` in the J.A.R.V.I.S repo.
2. **Second phrase:** edit `PHRASE` in the config cell above, then click that cell and
   use `Runtime → Run cell and below` (no need to re-run setup/data downloads).
3. Back on your machine, add the files to `wake_word.models` in `config/config.yaml`:

```yaml
wake_word:
  models: ["hey_jarvis", "models/wake/wake_up_mommys_home.onnx", "models/wake/jarvis_you_there.onnx"]
```
